In [9]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from tensorflow import keras

In [10]:
TARGET_COL = "Industry"
MIN_SAMPLES = 2       
RANDOM_STATE = 42

In [11]:
df = pd.read_csv("../data/agents_list.csv")
df["text"] = df["Use Case"].fillna("") + ". " + df["Description"].fillna("")

In [12]:
counts = df[TARGET_COL].value_counts()
keep = counts[counts >= MIN_SAMPLES].index
before = len(df)
df = df[df[TARGET_COL].isin(keep)].reset_index(drop=True)
print(f"{before - len(df)} satır, {MIN_SAMPLES}'den az örneği olan sınıflar yüzünden çıkarıldı")
print(f"Kalan satır: {len(df)}, sınıf sayısı: {df[TARGET_COL].nunique()}")

31 satır, 2'den az örneği olan sınıflar yüzünden çıkarıldı
Kalan satır: 40, sınıf sayısı: 17


In [13]:
y, classes = pd.factorize(df[TARGET_COL])
n_classes = len(classes)

In [14]:
X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    df["text"].values, y, test_size=0.2, random_state=RANDOM_STATE
)
X_train_txt, X_val_txt, y_train, y_val = train_test_split(
    X_train_txt, y_train, test_size=0.2, random_state=RANDOM_STATE
)

In [15]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
X_train = vectorizer.fit_transform(X_train_txt).toarray().astype("float32")
X_val = vectorizer.transform(X_val_txt).toarray().astype("float32")
X_test = vectorizer.transform(X_test_txt).toarray().astype("float32")

In [16]:
model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(n_classes, activation="softmax"),
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 128)                 │          62,976 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 17)                  │           1,105 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 72,337 (282.57 KB)

 Trainable params: 72,337 (282.57 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)
model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=300,
    batch_size=8,
    callbacks=[early_stop],
    verbose=0,
)


In [18]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
present = np.unique(np.concatenate([y_test, y_pred]))
print(classification_report(
    y_test, y_pred,
    labels=present,
    target_names=[str(classes[i]) for i in present],
    zero_division=0,
))


Test loss: 3.0195 | Test accuracy: 0.1250
                           precision    recall  f1-score   support

                   Gaming       0.00      0.00      0.00         1
            Cybersecurity       0.00      0.00      0.00         0
         Health insurance       0.00      0.00      0.00         0
             Productivity       0.00      0.00      0.00         0
              Recruitment       0.00      0.00      0.00         1
                   Travel       0.00      0.00      0.00         2
           AI Integration       0.00      0.00      0.00         1
            Collaboration       1.00      0.50      0.67         2
Supply Chain Optimization       0.00      0.00      0.00         0
         Tool Integration       0.00      0.00      0.00         1

                 accuracy                           0.12         8
                macro avg       0.10      0.05      0.07         8
             weighted avg       0.25      0.12      0.17         8



In [20]:
model.save("../model/ann_agent.keras")